In [3]:
import h5py
import os
import torch
import librosa
import numpy as np
import textgrid

def extract_and_save_features_to_hdf5(
    audio_dir,
    set_list,
    model,
    processor,
    output_h5_path="xie21_hubert_features_ft.h5",
    layers_spec=None
):
    if layers_spec is None:
        layers_spec = {"tr": [-1]}
        
    audio_paths = get_pathset(audio_dir) # 获取所有 wav 路径
    forward_module = getattr(model, "hubert", model)
    device = next(model.parameters()).device if next(model.parameters(), None) is not None else torch.device("cpu")
    model.to(device).eval()

    # 打开 HDF5 文件 (w 模式会覆盖已有文件)
    with h5py.File(output_h5_path, "w") as h5f:
        
        for each_path in audio_paths:
            # 假设你的文件名为 ALL_016_M_CMN_ENG_HT1.wav
            speaker_id = os.path.basename(each_path).replace(".wav", "")
            
            # 在 HDF5 中为这个 Speaker 创建一个文件夹 (Group)
            speaker_group = h5f.create_group(speaker_id)
            
            audio, sr = librosa.load(each_path, sr=None, mono=True)
            wave_res = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            tg = textgrid.TextGrid.fromFile(each_path[:-3] + "TextGrid")
            tg_sentence = tg[0]
            
            # (省略原有的 TextGrid 时间清理逻辑...)
            tg_sentence = [itv for itv in tg_sentence if itv.mark != ""]
            
            for s_idx in set_list:
                each_sentence = tg_sentence[s_idx]
                start = int(each_sentence.minTime * 16000)
                end   = int(each_sentence.maxTime * 16000)
                
                inputs = processor(wave_res[start:end], sampling_rate=16000, return_tensors="pt", padding=False)
                input_values = inputs.input_values.to(device)
                
                # 调用你的完美 hook 提取函数
                layer_feats = _extract_selected_layers(forward_module, input_values, layers_spec)
                
                # 为这个句子创建一个子文件夹 (Group)
                sentence_group = speaker_group.create_group(f"sentence_{s_idx:02d}")
                
                # 将该句子的各个层特征写入 HDF5 Dataset
                for layer_key, feat_np in layer_feats.items():
                    # feat_np shape: (T, D)
                    sentence_group.create_dataset(layer_key, data=feat_np, compression="gzip")
                    
            print(f"Finished processing and saving: {speaker_id}")
            
    print(f"🎉 All features saved successfully to {output_h5_path}")

In [4]:
import os
import glob
import h5py
import torch
import librosa
import numpy as np
import textgrid
import torch.nn.functional as F
from transformers import Wav2Vec2FeatureExtractor, HubertModel

# ====== 配置区 ======
AUDIO_DIR = r"../data/raw_data/xie21"
OUTPUT_H5 = "xie21_features_ft.h5"

set1_list = [0,1,2,3,4,5,6,7,8,9,10,12,13,14,15,16]
set2_list = [17,18,19,20,21,22,24,25,26,27,28,29,30,31,37,40]
combined_set = set1_list + set2_list

layers_spec = {
    "cnn": [2, 3, 4, 5, 6],
    "tr":  [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24], 
}
# ========================

def _register_cnn_hooks(hubert_model):
    store = {}
    handles = []
    conv_layers = hubert_model.feature_extractor.conv_layers
    def make_hook(i):
        def hook(module, inputs, output):
            store[i] = output.detach()
        return hook
    for i, layer in enumerate(conv_layers):
        handles.append(layer.register_forward_hook(make_hook(i)))
    return handles, store

def _cleanup_hooks(handles):
    for h in handles:
        try:
            h.remove()
        except:
            pass

def _extract_selected_layers(forward_module, input_values, layers_spec):
    cnn_handles, cnn_store = _register_cnn_hooks(forward_module)
    with torch.no_grad():
        out = forward_module(input_values, output_hidden_states=True, return_dict=True)
        tr_hidden = out.hidden_states
    _cleanup_hooks(cnn_handles)

    results = {}
    
    # 找到目标时间维度：取 Transformer 隐藏层的序列长度
    # out.hidden_states[0] 的形状是 (B, T_target, D)
    T_target = tr_hidden[0].size(1) 
    
    if "cnn" in layers_spec:
        for i in layers_spec["cnn"]:
            if i in cnn_store:
                feat = cnn_store[i]  # 原始形状: (B, C, T_current)
                
                # 【核心对齐逻辑】：使用一维自适应平均池化(Adaptive Avg Pool 1D)将 T_current 降采样到 T_target
                # 这样可以以最小的信息损失（局部能量平均），在时间轴上精确对齐所有的 CNN 层到统一形状
                if feat.size(2) != T_target:
                    feat = F.adaptive_avg_pool1d(feat, T_target)
                
                feat = feat.squeeze(0).permute(1, 0).contiguous() # 转为 (T_target, C)
                results[f"cnn_{i}"] = feat.cpu().numpy()
                
    if "tr" in layers_spec:
        for idx in layers_spec["tr"]:
            feat = tr_hidden[idx].squeeze(0) # (T_target, D)
            results[f"tr_{idx}"] = feat.cpu().numpy()
            
    return results

# ====== 执行逻辑 ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

processor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-large-ls960-ft")
model = HubertModel.from_pretrained("facebook/hubert-large-ls960-ft").to(device)
model.eval()

audio_paths = glob.glob(os.path.join(AUDIO_DIR, "*.wav"))
if not audio_paths:
    print("未找到音频文件，请检查 AUDIO_DIR 路径！")
else:
    with h5py.File(OUTPUT_H5, "w") as h5f:
        for each_path in audio_paths:
            speaker_id = os.path.basename(each_path).replace(".wav", "")
            speaker_group = h5f.create_group(speaker_id)
            
            audio, sr = librosa.load(each_path, sr=None, mono=True)
            wave_res = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            tg = textgrid.TextGrid.fromFile(each_path[:-3] + "TextGrid")
            tg_sentence = tg[0]
            
            for idx, itv in enumerate(tg_sentence):
                if itv.mark != "":
                    tg_sentence[idx - 1].maxTime = tg_sentence[idx].minTime
            tg_sentence = [itv for itv in tg_sentence if itv.mark != ""]
            
            for s_idx in combined_set:
                each_sentence = tg_sentence[s_idx]
                start = int(each_sentence.minTime * 16000)
                end   = int(each_sentence.maxTime * 16000)
                
                inputs = processor(wave_res[start:end], sampling_rate=16000, return_tensors="pt", padding=False)
                input_values = inputs.input_values.to(device)
                
                layer_feats = _extract_selected_layers(model, input_values, layers_spec)
                
                sentence_group = speaker_group.create_group(f"sentence_{s_idx:02d}")
                for layer_key, feat_np in layer_feats.items():
                    sentence_group.create_dataset(layer_key, data=feat_np, compression="gzip")
                    
            print(f"Processed speaker: {speaker_id}")
    print("✅ Feature extraction completed!")


Using device: cuda


c:\Users\Alex\anaconda3\envs\BayesPCN\lib\site-packages\transformers\models\hubert\modeling_hubert.py:762: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Processed speaker: ALL_016_M_CMN_ENG_HT1
Processed speaker: ALL_021_M_CMN_ENG_HT1
Processed speaker: ALL_032_M_CMN_ENG_HT1
Processed speaker: ALL_035_M_CMN_ENG_HT1
Processed speaker: ALL_037_M_CMN_ENG_HT1
Processed speaker: ALL_043_M_CMN_ENG_HT1
Processed speaker: ALL_055_M_ENG_ENG_HT1
Processed speaker: ALL_066_M_ENG_ENG_HT1
Processed speaker: ALL_070_M_ENG_ENG_HT1
Processed speaker: ALL_131_M_ENG_ENG_HT1
Processed speaker: ALL_133_M_ENG_ENG_HT1
✅ Feature extraction completed!


In [5]:
import h5py
import numpy as np
from joblib import Parallel, delayed
from sklearn.manifold import TSNE
import multiprocessing
import time

INPUT_H5 = "xie21_features_ft.h5"
OUTPUT_H5 = "xie21_tsne_3d_ft.h5"
TSNE_DIM = 3

def get_all_layers(h5_path):
    layers = set()
    with h5py.File(h5_path, "r") as h5f:
        first_speaker = list(h5f.keys())[0]
        first_sentence = list(h5f[first_speaker].keys())[0]
        for layer_key in h5f[first_speaker][first_sentence].keys():
            layers.add(layer_key)
    return list(layers)

def process_single_layer(layer_name):
    """提取单个层级的数据并运行 t-SNE (joblib的worker任务)"""
    all_frames = []
    metadata = [] 
    
    with h5py.File(INPUT_H5, "r") as h5f:
        for speaker_id in h5f.keys():
            for sentence_id in h5f[speaker_id].keys():
                if layer_name in h5f[speaker_id][sentence_id]:
                    feat_matrix = h5f[speaker_id][sentence_id][layer_name][:]
                    T = feat_matrix.shape[0]
                    all_frames.append(feat_matrix)
                    metadata.append({"speaker": speaker_id, "sentence": sentence_id, "length": T})
                    
    if not all_frames:
        return layer_name, None, metadata

    flatten = np.vstack(all_frames)
    print(f"[{layer_name}] 开始计算 t-SNE... 帧数: {flatten.shape[0]}, 特征维度: {flatten.shape[1]}")
    start_time = time.time()
    
    tsne = TSNE(n_components=TSNE_DIM, random_state=42, n_jobs=1)
    reduced_features = tsne.fit_transform(flatten)
    
    cost_time = time.time() - start_time
    print(f"[{layer_name}] t-SNE 计算完成! 耗时: {cost_time:.2f} 秒")
    
    # 将降维结果一起返回给主进程
    return layer_name, reduced_features, metadata

# ========= 启动 joblib 多进程 TSNE =========
layers = get_all_layers(INPUT_H5)
print(f"需处理的层: {layers}")

# 依然建议设置一个合理的并行数防止内存爆炸 (t-SNE还是很耗内存的)
max_workers = min(multiprocessing.cpu_count(), 24)
print(f"🔥 启动 joblib 并发！分配的 CPU 核心数: {max_workers}")

# 【核心修改】使用优雅且稳健的 joblib 并发
# n_jobs 指定并发数，默认会使用非常安全的 loky 后端
results = Parallel(n_jobs=max_workers)(
    delayed(process_single_layer)(layer) for layer in layers
)

# 等待所有层计算完毕后，统一写入结果 HDF5
print(f"\n正在将所有结果汇总写入 {OUTPUT_H5}...")
with h5py.File(OUTPUT_H5, "w") as out_h5:
    for layer_name, reduced_features, metadata in results:
        if reduced_features is None: 
            continue
        
        layer_group = out_h5.create_group(layer_name)
        current_idx = 0
        for meta in metadata:
            spk, sen, length = meta["speaker"], meta["sentence"], meta["length"]
            sentence_3d = reduced_features[current_idx : current_idx + length]
            current_idx += length
            
            if spk not in layer_group:
                layer_group.create_group(spk)
            layer_group[spk].create_dataset(sen, data=sentence_3d)

print(f"🎉 完美收工！结果已存储至 {OUTPUT_H5}")

需处理的层: ['cnn_5', 'cnn_2', 'tr_20', 'tr_8', 'tr_12', 'tr_10', 'tr_16', 'cnn_6', 'tr_18', 'cnn_3', 'tr_4', 'tr_22', 'tr_14', 'tr_6', 'tr_2', 'tr_0', 'tr_24', 'cnn_4']
🔥 启动 joblib 并发！分配的 CPU 核心数: 24

正在将所有结果汇总写入 xie21_tsne_3d_ft.h5...
🎉 完美收工！结果已存储至 xie21_tsne_3d_ft.h5
